In [ ]:
"""
SELMA (Spike-input Extreme Learning Machine) for a NONLINEAR, NON-STATIONARY HFT
sub-problem: short-horizon return-DIRECTION prediction from spike-encoded order-flow
features, with regime shifts.

SELMA = a fixed RANDOM nonlinear projection of the (spike-count) inputs + a linear
readout solved in CLOSED FORM (ridge). Two properties matter:
  * nonlinear  -> beats a linear/Kalman-class baseline when the map is nonlinear
  * closed-form readout -> INSTANT retraining -> fast recalibration under regime shift

We compare on ACCURACY, INFERENCE latency, and -- the real question -- RETRAINING latency,
against: linear (logistic), a backprop MLP, and gradient boosting (strong nonlinear ML).
"""
import time, numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier

D = 6                       # underlying order-flow features
K = 6.0                     # spike-encoding gain


def make_data(n, regime_sign, seed):
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, D))
    # genuinely NONLINEAR return: concave impact + interaction + regime-dependent + threshold
    r = (0.9 * np.sign(x[:, 0]) * np.sqrt(np.abs(x[:, 0]))
         + 0.8 * x[:, 1] * x[:, 2]
         + regime_sign * 0.7 * np.tanh(2 * x[:, 3])
         + 0.5 * (x[:, 4] > 0.5) * x[:, 5]
         + 0.5 * rng.standard_normal(n))
    y = (r > 0).astype(int)
    # SPIKE-INPUT encoding: signed rate code -> nonnegative counts, 2*D channels
    pos = rng.poisson(K * np.clip(x, 0, None)); neg = rng.poisson(K * np.clip(-x, 0, None))
    S = np.hstack([pos, neg]).astype(np.float64)
    return S, y

def real_data():
  ""
  return 0



class SELMA:
    """Spike-input ELM: standardize -> random tanh projection (fixed) -> ridge readout."""
    def __init__(self, n_in, n_hidden=300, lam=5.0, seed=0):
        r = np.random.default_rng(seed)
        self.Win = r.standard_normal((n_in, n_hidden)) / np.sqrt(n_in)
        self.b = r.standard_normal(n_hidden); self.lam = lam
    def _h(self, S):
        Z = (S - self.mu) / self.sd
        return np.tanh(Z @ self.Win + self.b)
    def fit(self, S, y):                       # CLOSED FORM -> instant (re)training
        self.mu = S.mean(0); self.sd = S.std(0) + 1e-6
        H = self._h(S); t = 2 * y - 1          # +/-1 target
        A = H.T @ H + self.lam * np.eye(H.shape[1])
        self.beta = np.linalg.solve(A, H.T @ t)
    def predict(self, S): return (self._h(S) @ self.beta > 0).astype(int)


# minimal backprop MLP (for a fair "trained nonlinear net" + its retraining cost)
class MLP:
    def __init__(self, n_in, n_hid=64, seed=0):
        r = np.random.default_rng(seed)
        self.W1 = r.standard_normal((n_in, n_hid)) * np.sqrt(2 / n_in); self.b1 = np.zeros(n_hid)
        self.W2 = r.standard_normal((n_hid, 2)) * np.sqrt(2 / n_hid); self.b2 = np.zeros(2)
    def _fwd(self, X):
        self.mu = getattr(self, "mu", X.mean(0)); self.sd = getattr(self, "sd", X.std(0) + 1e-6)
        Z = (X - self.mu) / self.sd; h = np.maximum(Z @ self.W1 + self.b1, 0)
        o = h @ self.W2 + self.b2; return Z, h, o
    def fit(self, X, y, epochs=60, lr=0.05, bs=256):
        self.mu = X.mean(0); self.sd = X.std(0) + 1e-6
        for _ in range(epochs):
            idx = np.random.permutation(len(X))
            for i in range(0, len(X), bs):
                b = idx[i:i + bs]; Z, h, o = self._fwd(X[b])
                p = np.exp(o - o.max(1, keepdims=True)); p /= p.sum(1, keepdims=True)
                g = p; g[np.arange(len(b)), y[b]] -= 1; g /= len(b)
                gW2 = h.T @ g; gb2 = g.sum(0); gh = (g @ self.W2.T) * (h > 0)
                gW1 = Z.T @ gh; gb1 = gh.sum(0)
                self.W2 -= lr * gW2; self.b2 -= lr * gb2; self.W1 -= lr * gW1; self.b1 -= lr * gb1
    def predict(self, X): return self._fwd(X)[2].argmax(1)


def bench(model, Ste):
    t = time.perf_counter(); p = model.predict(Ste); dt = (time.perf_counter() - t) / len(Ste) * 1e6
    return p, dt


def main():
    Str, ytr = make_data(8000, +1, 0); Ste, yte = make_data(3000, +1, 1)
    nin = Str.shape[1]
    print(f"nonlinear HFT direction task | spike channels={nin} | chance=0.500\n")
    print(f"{'model':22s}{'accuracy':>9s}{'infer_us':>10s}{'retrain_ms':>12s}")
    res = {}
    for name, mk, fit in [
        ("linear (logistic)", lambda: LogisticRegression(max_iter=200), "sk"),
        ("SELMA (spike-ELM)", lambda: SELMA(nin, 300, 5.0, 0), "own"),
        ("MLP (backprop)", lambda: MLP(nin, 64, 0), "own"),
        ("gradient boosting", lambda: GradientBoostingClassifier(n_estimators=150, max_depth=3), "sk"),
    ]:
        m = mk()
        t = time.perf_counter(); m.fit(Str, ytr); rt = (time.perf_counter() - t) * 1e3
        p, it = bench(m, Ste); acc = float((p == yte).mean())
        res[name] = (acc, it, rt); print(f"{name:22s}{acc:9.3f}{it:10.2f}{rt:12.1f}")

    # --- non-stationarity: regime flips; measure accuracy STATIC vs RETRAINED + retrain cost ---
    print("\nregime shift (nonlinear term flips sign) -- adapt by retraining:")
    Sr, yr = make_data(3000, -1, 2)                    # new regime test set
    print(f"{'model':22s}{'static acc':>11s}{'retrained acc':>14s}{'retrain_ms':>12s}")
    for name, mk in [("SELMA (spike-ELM)", lambda: SELMA(nin, 300, 5.0, 0)),
                     ("MLP (backprop)", lambda: MLP(nin, 64, 0))]:
        m = mk(); m.fit(Str, ytr)
        static = float((m.predict(Sr) == yr).mean())
        Sr2, yr2 = make_data(6000, -1, 3)              # fresh regime data to retrain on
        t = time.perf_counter(); m.fit(Sr2, yr2); rt = (time.perf_counter() - t) * 1e3
        retr = float((m.predict(Sr) == yr).mean())
        print(f"{name:22s}{static:11.3f}{retr:14.3f}{rt:12.1f}")

    np.savez("/home/",
             names=list(res.keys()), vals=np.array(list(res.values())))


if __name__ == "__main__":
    main()

nonlinear HFT direction task | spike channels=12 | chance=0.500

model                  accuracy  infer_us  retrain_ms
linear (logistic)         0.779      0.21        38.9
SELMA (spike-ELM)         0.842     42.44       367.3
MLP (backprop)            0.841      3.06      1336.6
gradient boosting         0.826      3.00      1902.4

regime shift (nonlinear term flips sign) -- adapt by retraining:
model                  static acc retrained acc  retrain_ms
SELMA (spike-ELM)           0.675         0.825       145.7
MLP (backprop)              0.674         0.836       649.5
